In [2]:
# ============================================
# CELL 1: SETUP
# ============================================

import os
import pandas as pd
import duckdb
from datetime import datetime

# ⭐ UPDATE THIS PATH IF NEEDED ⭐
BASE = r"D:\abc4\Pneumonia-mimic iv"

MIMIC_ROOT      = os.path.join(BASE, "mimic-iv-3.1")
HOSP_PATH       = os.path.join(MIMIC_ROOT, "hosp")
ICU_PATH        = os.path.join(MIMIC_ROOT, "icu")

CLEANED_DATASET = os.path.join(BASE, "pneumonia_dataset_final.csv")
D_LABITEMS_FILE = os.path.join(HOSP_PATH, "d_labitems.csv")
LABEVENTS_FILE  = os.path.join(HOSP_PATH, "labevents.csv")

print("=" * 60)
print("📁 PATH VERIFICATION")
print("=" * 60)

for name, path in [
    ("Cleaned dataset", CLEANED_DATASET),
    ("d_labitems.csv",  D_LABITEMS_FILE),
    ("labevents.csv",   LABEVENTS_FILE),
]:
    if os.path.exists(path):
        size_mb = os.path.getsize(path) / (1024**2)
        print(f"✅ {name}: {size_mb:.1f} MB")
    else:
        print(f"❌ {name}: NOT FOUND → {path}")

📁 PATH VERIFICATION
✅ Cleaned dataset: 2.6 MB
✅ d_labitems.csv: 0.1 MB
✅ labevents.csv: 17550.3 MB


In [3]:
# ============================================
# CELL 2: VERIFY LAB ITEMIDS
# ============================================

print("=" * 60)
print("🔍 FINDING CORRECT LAB ITEMIDS")
print("=" * 60)

query = f"""
SELECT itemid, label, fluid, category
FROM read_csv_auto('{D_LABITEMS_FILE}')
WHERE LOWER(label) LIKE '%white blood cell%'
   OR LOWER(label) LIKE '%wbc%'
   OR LOWER(label) LIKE '%neutrophil%'
   OR LOWER(label) LIKE '%lymphocyte%'
   OR LOWER(label) LIKE '%c-reactive%'
   OR LOWER(label) LIKE '%c reactive%'
   OR LOWER(label) LIKE '%crp%'
   OR LOWER(label) LIKE '%procalcitonin%'
ORDER BY label
"""

lab_items = duckdb.sql(query).df()
print(lab_items.to_string())

🔍 FINDING CORRECT LAB ITEMIDS
    itemid                                 label                fluid    category
0    51133             Absolute Lymphocyte Count                Blood  Hematology
1    53157             Absolute Lymphocyte Count                Blood   Chemistry
2    52769             Absolute Lymphocyte Count                Blood  Hematology
3    53159             Absolute Neutrophil Count                Blood   Chemistry
4    52075             Absolute Neutrophil Count                Blood  Hematology
5    50872  Anti-Neutrophil Cytoplasmic Antibody                Blood   Chemistry
6    51440                  Atypical Lymphocytes              Pleural  Hematology
7    51110                  Atypical Lymphocytes              Ascites  Hematology
8    51143                  Atypical Lymphocytes                Blood  Hematology
9    52221                  Atypical Lymphocytes  Cerebrospinal Fluid  Hematology
10   51385                  Atypical Lymphocytes     Other Body Flui

In [4]:
# ============================================
# EXTRA SEARCH FOR PROCALCITONIN
# ============================================

query = f"""
SELECT itemid, label, fluid, category
FROM read_csv_auto('{D_LABITEMS_FILE}')
WHERE LOWER(label) LIKE '%pct%'
   OR LOWER(label) LIKE '%calcitonin%'
   OR LOWER(label) LIKE '%sepsis%'
ORDER BY label
"""

pct_items = duckdb.sql(query).df()
print(pct_items.to_string())

Empty DataFrame
Columns: [itemid, label, fluid, category]
Index: []


In [5]:
# ============================================
# COMPLETE LAB EXTRACTION PIPELINE
# ============================================

import os
import pandas as pd
import numpy as np
import duckdb
from datetime import datetime

# ⭐ PATHS ⭐
BASE            = r"D:\abc4\Pneumonia-mimic iv"
MIMIC_ROOT      = os.path.join(BASE, "mimic-iv-3.1")
HOSP_PATH       = os.path.join(MIMIC_ROOT, "hosp")

CLEANED_DATASET = os.path.join(BASE, "pneumonia_dataset_final.csv")
LABEVENTS_FILE  = os.path.join(HOSP_PATH, "labevents.csv")

# ⭐ VERIFIED ITEMIDS (from your search) ⭐
LAB_ITEMIDS = {
    # WBC
    51300: 'wbc',
    51301: 'wbc',
    51756: 'wbc',
    51755: 'wbc',
    # Neutrophils
    51256: 'neutrophils',
    53164: 'neutrophils',
    # Lymphocytes
    51244: 'lymphocytes',
    53188: 'lymphocytes',
    # CRP
    50889: 'crp',
    51652: 'crp',
}

print("=" * 60)
print("🧪 LAB EXTRACTION PIPELINE")
print("=" * 60)
print(f"Start: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

# ── Step 1: Load cleaned dataset ──
print("\n📂 Loading cleaned dataset...")
df = pd.read_csv(CLEANED_DATASET)
print(f"   ✅ Loaded: {df.shape}")
print(f"   Patients: {len(df)}")
all_hadm_ids = df['hadm_id'].tolist()

# ── Step 2: Extract labs in batches ──
print("\n🧪 Extracting labs from 18 GB labevents.csv...")
BATCH_SIZE = 500
all_labs = []
total_batches = (len(all_hadm_ids) // BATCH_SIZE) + 1

for i in range(0, len(all_hadm_ids), BATCH_SIZE):
    batch = all_hadm_ids[i:i+BATCH_SIZE]
    batch_str = ','.join(map(str, batch))
    itemids_str = ','.join(map(str, LAB_ITEMIDS.keys()))
    
    print(f"   Batch {i//BATCH_SIZE + 1}/{total_batches}")
    
    q = f"""
    SELECT hadm_id, itemid, valuenum
    FROM read_csv_auto('{LABEVENTS_FILE}')
    WHERE hadm_id IN ({batch_str})
      AND itemid IN ({itemids_str})
      AND valuenum IS NOT NULL
      AND valuenum > 0
    """
    res = duckdb.sql(q).df()
    if not res.empty:
        all_labs.append(res)

if not all_labs:
    print("\n❌ No lab measurements extracted!")
    raise SystemExit("No labs found — check LAB_ITEMIDS and LABEVENTS_FILE path")

labs_df = pd.concat(all_labs, ignore_index=True)
print(f"\n✅ Extracted {len(labs_df):,} lab measurements")
print(f"   Unique patients with labs: {labs_df['hadm_id'].nunique():,}")

# ── Step 3: Pivot to one row per patient ──
print("\n🔄 Pivoting labs...")
labs_df['lab'] = labs_df['itemid'].map(LAB_ITEMIDS)

labs_pivot = (
    labs_df.groupby(['hadm_id', 'lab'])['valuenum']
    .first()
    .unstack()
    .reset_index()
)
print(f"   ✅ Pivoted shape: {labs_pivot.shape}")
print(f"   Columns: {list(labs_pivot.columns)}")

# Availability
print(f"\n📊 Lab Availability (out of {len(labs_pivot)}):")
for lab in ['wbc', 'neutrophils', 'lymphocytes', 'crp']:
    if lab in labs_pivot.columns:
        n = labs_pivot[lab].notna().sum()
        pct = n / len(labs_pivot) * 100
        print(f"   {lab}: {n} patients ({pct:.1f}%)")
    else:
        print(f"   {lab}: NOT FOUND ❌")

# ── Step 4: Merge with cleaned dataset ──
print("\n🔗 Merging with cleaned dataset...")
df_merged = df.merge(labs_pivot, on='hadm_id', how='left')
print(f"   ✅ Merged shape: {df_merged.shape}")

# ── Step 5: Create value + indicator pairs ──
print("\n🧩 Creating value + indicator columns...")
for lab in ['wbc', 'neutrophils', 'lymphocytes', 'crp']:
    if lab in df_merged.columns:
        missing = df_merged[lab].isnull().sum()
        pct = missing / len(df_merged) * 100
        
        df_merged[f'{lab}_done']   = df_merged[lab].notna().astype(int)
        df_merged[f'{lab}_value']  = df_merged[lab].fillna(0)
        df_merged = df_merged.drop(lab, axis=1)
        
        print(f"   ✅ {lab}: {missing} missing ({pct:.1f}%) → value + done")

print(f"\n✅ Final shape: {df_merged.shape}")

# ── Step 6: Save ──
print("\n💾 Saving datasets...")
research_path = os.path.join(BASE, "pneumonia_dataset_with_labs.csv")
df_merged.to_csv(research_path, index=False)
print(f"   ✅ Research (with labs): {research_path}")

lab_cols = [c for c in df_merged.columns if '_value' in c or '_done' in c]
df_app = df_merged.drop(columns=lab_cols)
app_path = os.path.join(BASE, "pneumonia_dataset_no_labs.csv")
df_app.to_csv(app_path, index=False)
print(f"   ✅ Application (no labs): {app_path}")

print(f"\n{'='*60}")
print("✅ LAB EXTRACTION COMPLETE!")
print(f"End: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"{'='*60}")
print(f"\nNew columns added:")
for c in df_merged.columns:
    if '_value' in c or '_done' in c:
        print(f"   - {c}")

🧪 LAB EXTRACTION PIPELINE
Start: 2026-10-14 12:11:22

📂 Loading cleaned dataset...
   ✅ Loaded: (7647, 40)
   Patients: 7647

🧪 Extracting labs from 18 GB labevents.csv...
   Batch 1/16


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   Batch 2/16


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   Batch 3/16


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

... (23 outputs left out)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   Batch 16/16


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


✅ Extracted 184,059 lab measurements
   Unique patients with labs: 7,592

🔄 Pivoting labs...
   ✅ Pivoted shape: (7592, 5)
   Columns: ['hadm_id', 'crp', 'lymphocytes', 'neutrophils', 'wbc']

📊 Lab Availability (out of 7592):
   wbc: 7592 patients (100.0%)
   neutrophils: 5531 patients (72.9%)
   lymphocytes: 5523 patients (72.7%)
   crp: 993 patients (13.1%)

🔗 Merging with cleaned dataset...
   ✅ Merged shape: (7647, 44)

🧩 Creating value + indicator columns...
   ✅ wbc: 55 missing (0.7%) → value + done
   ✅ neutrophils: 2116 missing (27.7%) → value + done
   ✅ lymphocytes: 2124 missing (27.8%) → value + done
   ✅ crp: 6654 missing (87.0%) → value + done

✅ Final shape: (7647, 48)

💾 Saving datasets...
   ✅ Research (with labs): D:\abc4\Pneumonia-mimic iv\pneumonia_dataset_with_labs.csv
   ✅ Application (no labs): D:\abc4\Pneumonia-mimic iv\pneumonia_dataset_no_labs.csv

✅ LAB EXTRACTION COMPLETE!
End: 2026-10-14 12:17:25

New columns added:
   - wbc_done
   - wbc_value
   - neutrop

In [6]:
# ============================================
# VERIFY WBC IS REAL
# ============================================

import pandas as pd

df = pd.read_csv(r"D:\abc4\Pneumonia-mimic iv\pneumonia_dataset_with_labs.csv")

print("WBC value distribution:")
print(df['wbc_value'].describe())

print(f"\nWBC == 0 (test not done): {(df['wbc_value'] == 0).sum()}")
print(f"WBC == 0 but done=1:      {((df['wbc_value'] == 0) & (df['wbc_done'] == 1)).sum()}")
print(f"WBC done=1:               {df['wbc_done'].sum()}")

# Check for strange patterns
print(f"\nWBC range: {df[df['wbc_done']==1]['wbc_value'].min():.1f} - {df[df['wbc_done']==1]['wbc_value'].max():.1f}")

WBC value distribution:
count    7647.000000
mean       12.142946
std        10.537245
min         0.000000
25%         7.200000
50%        10.400000
75%        14.900000
max       471.700000
Name: wbc_value, dtype: float64

WBC == 0 (test not done): 55
WBC == 0 but done=1:      0
WBC done=1:               7592

WBC range: 0.1 - 471.7


In [7]:
# ============================================
# VERIFY LABS BELONG TO CLEANED DATASET PATIENTS
# ============================================

import os
import pandas as pd
import duckdb

BASE = r"D:\abc4\Pneumonia-mimic iv"
MIMIC_ROOT = os.path.join(BASE, "mimic-iv-3.1")
HOSP_PATH = os.path.join(MIMIC_ROOT, "hosp")
LABEVENTS_FILE = os.path.join(HOSP_PATH, "labevents.csv")

CLEANED = os.path.join(BASE, "pneumonia_dataset_final.csv")
WITH_LABS = os.path.join(BASE, "pneumonia_dataset_with_labs.csv")

print("=" * 60)
print("🔍 LAB-MEMBERSHIP VERIFICATION")
print("=" * 60)

# ── 1. Load cleaned dataset patients ──
df_clean = pd.read_csv(CLEANED)
clean_hadm = set(df_clean['hadm_id'].unique())
print(f"\n📂 Cleaned dataset:")
print(f"   Unique hadm_id: {len(clean_hadm):,}")

# ── 2. Load dataset with labs ──
df_labs = pd.read_csv(WITH_LABS)
labs_hadm = set(df_labs['hadm_id'].unique())
print(f"\n📂 Dataset with labs:")
print(f"   Unique hadm_id: {len(labs_hadm):,}")

# ── 3. Are they identical? ──
print(f"\n🔍 Set comparison:")
print(f"   In cleaned but NOT in labs: {len(clean_hadm - labs_hadm)}")
print(f"   In labs but NOT in cleaned: {len(labs_hadm - clean_hadm)}")

if clean_hadm == labs_hadm:
    print(f"   ✅ PASS: Both datasets cover exactly the same patients")
else:
    print(f"   ❌ FAIL: Patient mismatch detected")

# ── 4. Check each row in df_labs actually corresponds to cleaned ──
print(f"\n🔍 Row-level check:")
rows_in_clean = df_labs['hadm_id'].isin(clean_hadm).sum()
print(f"   Rows in with_labs that exist in cleaned: {rows_in_clean:,} / {len(df_labs):,}")

if rows_in_clean == len(df_labs):
    print(f"   ✅ PASS: Every row maps to a cleaned-dataset patient")
else:
    print(f"   ❌ FAIL: {len(df_labs) - rows_in_clean} rows are from other patients")

# ── 5. Verify lab values (from raw labevents.csv) match ──
print(f"\n🔍 Cross-checking lab values against raw labevents.csv...")

# Take 5 random patients from cleaned dataset
sample_hadm = list(clean_hadm)[:5]
sample_str = ','.join(map(str, sample_hadm))

q = f"""
SELECT hadm_id, itemid, valuenum
FROM read_csv_auto('{LABEVENTS_FILE}')
WHERE hadm_id IN ({sample_str})
  AND itemid IN (51300, 51301, 51756, 51755)
  AND valuenum IS NOT NULL AND valuenum > 0
LIMIT 5
"""
sample_raw = duckdb.sql(q).df()
print("\n   Raw labevents.csv samples:")
print(sample_raw.to_string(index=False))

print("\n   Our extracted dataset samples:")
print(df_labs[df_labs['hadm_id'].isin(sample_hadm)][
    ['hadm_id', 'wbc_value', 'wbc_done']
].head(5).to_string(index=False))

# ── 6. Confirm no extra hadm_ids snuck in ──
extra = labs_hadm - clean_hadm
if extra:
    print(f"\n⚠️ WARNING: {len(extra)} extra hadm_ids found in with_labs")
    print(f"   Examples: {list(extra)[:5]}")
else:
    print(f"\n✅ No extra patients — labs only belong to cleaned dataset")

print("\n" + "=" * 60)
print("✅ VERIFICATION COMPLETE")
print("=" * 60)

🔍 LAB-MEMBERSHIP VERIFICATION

📂 Cleaned dataset:
   Unique hadm_id: 7,647

📂 Dataset with labs:
   Unique hadm_id: 7,647

🔍 Set comparison:
   In cleaned but NOT in labs: 0
   In labs but NOT in cleaned: 0
   ✅ PASS: Both datasets cover exactly the same patients

🔍 Row-level check:
   Rows in with_labs that exist in cleaned: 7,647 / 7,647
   ✅ PASS: Every row maps to a cleaned-dataset patient

🔍 Cross-checking lab values against raw labevents.csv...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [8]:
# ============================================
# STRONG VERIFICATION — ROW-LEVEL MATCHING
# ============================================

import os
import pandas as pd
import numpy as np
import duckdb

BASE = r"D:\abc4\Pneumonia-mimic iv"
MIMIC_ROOT = os.path.join(BASE, "mimic-iv-3.1")
HOSP_PATH = os.path.join(MIMIC_ROOT, "hosp")
LABEVENTS_FILE = os.path.join(HOSP_PATH, "labevents.csv")

CLEANED  = os.path.join(BASE, "pneumonia_dataset_final.csv")
WITH_LABS = os.path.join(BASE, "pneumonia_dataset_with_labs.csv")

print("=" * 60)
print("🔍 STRONG LAB-MEMBERSHIP VERIFICATION")
print("=" * 60)

# ── Load datasets ──
df_clean = pd.read_csv(CLEANED)
df_labs  = pd.read_csv(WITH_LABS)

# ============================================
# CHECK 1: Merge-based row correspondence
# ============================================
print("\n🔍 CHECK 1: Row-by-row correspondence via merge")
print("-" * 60)

# Merge on hadm_id; if rows align 1:1, we'll see identical lengths
merged = df_clean.merge(
    df_labs,
    on='hadm_id',
    how='outer',
    indicator=True,
    suffixes=('_clean', '_labs')
)

print(f"   Outer-merge result: {len(merged)} rows")
print(f"   Both-sides matched: {(merged['_merge'] == 'both').sum()}")
print(f"   Left-only (in clean only): {(merged['_merge'] == 'left_only').sum()}")
print(f"   Right-only (in labs only): {(merged['_merge'] == 'right_only').sum()}")

if (merged['_merge'] == 'both').sum() == len(df_clean) == len(df_labs):
    print("   ✅ PASS: Every patient exists in both datasets")
else:
    print("   ❌ FAIL: Row mismatch detected")

# ============================================
# CHECK 2: Label consistency (same patient → same target)
# ============================================
print("\n🔍 CHECK 2: Target label consistency")
print("-" * 60)

label_check = df_clean[['hadm_id', 'pneumonia_label']].merge(
    df_labs[['hadm_id', 'pneumonia_label']],
    on='hadm_id',
    suffixes=('_clean', '_labs')
)

mismatch = (label_check['pneumonia_label_clean'] != label_check['pneumonia_label_labs']).sum()
print(f"   Total patients: {len(label_check)}")
print(f"   Label mismatches: {mismatch}")

if mismatch == 0:
    print("   ✅ PASS: Every patient has the same pneumonia_label in both files")
else:
    print(f"   ❌ FAIL: {mismatch} patients have different labels!")
    # Show mismatches
    bad = label_check[label_check['pneumonia_label_clean'] != label_check['pneumonia_label_labs']]
    print(bad.head())

# ============================================
# CHECK 3: Compare physiological features (should be IDENTICAL)
# ============================================
print("\n🔍 CHECK 3: Physiological feature values match across files")
print("-" * 60)

# These columns exist in BOTH files and should be identical
common_cols = [
    'heart_rate_mean', 'spo2_mean', 'respiratory_rate_mean',
    'temperature_mean', 'sbp_mean'
]

for col in common_cols:
    if col in df_clean.columns and col in df_labs.columns:
        # Merge on hadm_id and compare
        cmp = df_clean[['hadm_id', col]].merge(
            df_labs[['hadm_id', col]],
            on='hadm_id',
            suffixes=('_clean', '_labs')
        )
        diffs = (cmp[f'{col}_clean'] != cmp[f'{col}_labs']).sum()
        status = "✅" if diffs == 0 else "❌"
        print(f"   {status} {col}: {diffs} mismatches out of {len(cmp)}")

# ============================================
# CHECK 4: Sample 5 patients — trace from raw file to final
# ============================================
print("\n🔍 CHECK 4: Trace 5 patients from raw labevents.csv")
print("-" * 60)

sample_ids = df_labs['hadm_id'].head(5).tolist()
sample_str = ','.join(map(str, sample_ids))

# Get all WBC measurements for these patients
q = f"""
SELECT hadm_id, itemid, valuenum, charttime
FROM read_csv_auto('{LABEVENTS_FILE}')
WHERE hadm_id IN ({sample_str})
  AND itemid IN (51300, 51301, 51755, 51756)
  AND valuenum IS NOT NULL AND valuenum > 0
ORDER BY hadm_id, charttime
"""
raw = duckdb.sql(q).df()

print("\n   📊 Raw WBC measurements (from labevents.csv):")
for pid in sample_ids:
    patient_raw = raw[raw['hadm_id'] == pid]
    if len(patient_raw) > 0:
        first_row = patient_raw.iloc[0]
        print(f"\n   hadm_id = {pid}")
        print(f"      Total WBC measurements: {len(patient_raw)}")
        print(f"      First measurement: {first_row['valuenum']} at {first_row['charttime']}")
        print(f"      Last measurement:  {patient_raw.iloc[-1]['valuenum']} at {patient_raw.iloc[-1]['charttime']}")
        print(f"      Min / Max: {patient_raw['valuenum'].min()} / {patient_raw['valuenum'].max()}")
    else:
        print(f"\n   hadm_id = {pid}: no WBC found")

print("\n   📊 Our extracted dataset (wbc_value):")
for pid in sample_ids:
    row = df_labs[df_labs['hadm_id'] == pid].iloc[0]
    print(f"      hadm_id = {pid}: wbc_value = {row['wbc_value']}, wbc_done = {row['wbc_done']}")

# ============================================
# CHECK 5: Re-derive wbc_value independently and compare
# ============================================
print("\n🔍 CHECK 5: Independently re-extract and compare")
print("-" * 60)

# Re-extract WBC for ALL patients (using different method: just min value)
q2 = f"""
SELECT hadm_id, MIN(valuenum) as wbc_min
FROM read_csv_auto('{LABEVENTS_FILE}')
WHERE hadm_id IN ({','.join(map(str, df_labs['hadm_id'].tolist()[:1000]))})
  AND itemid IN (51300, 51301, 51755, 51756)
  AND valuenum IS NOT NULL AND valuenum > 0
GROUP BY hadm_id
LIMIT 1000
"""
recheck = duckdb.sql(q2).df()

# Compare with our saved values (only for 1000 patients)
compare = df_labs[df_labs['hadm_id'].isin(recheck['hadm_id'])][['hadm_id', 'wbc_value']].merge(
    recheck, on='hadm_id'
)

print(f"   Compared {len(compare)} patients")
print(f"   Our wbc_value = MIN(wbc): {(compare['wbc_value'] == compare['wbc_min']).sum()}")
print(f"   Our wbc_value > MIN(wbc): {(compare['wbc_value'] > compare['wbc_min']).sum()}")
print(f"   Our wbc_value < MIN(wbc): {(compare['wbc_value'] < compare['wbc_min']).sum()}")

# Note: our value should be >= MIN (since we took first, not min)
if (compare['wbc_value'] < compare['wbc_min']).sum() == 0:
    print("   ✅ PASS: Our wbc_value is never below the true minimum")
    print("           (first-value is always >= min — logically consistent)")
else:
    print("   ❌ FAIL: Some wbc_value is below the true minimum — impossible!")

print("\n" + "=" * 60)
print("✅ VERIFICATION COMPLETE")
print("=" * 60)

🔍 STRONG LAB-MEMBERSHIP VERIFICATION

🔍 CHECK 1: Row-by-row correspondence via merge
------------------------------------------------------------
   Outer-merge result: 7647 rows
   Both-sides matched: 7647
   Left-only (in clean only): 0
   Right-only (in labs only): 0
   ✅ PASS: Every patient exists in both datasets

🔍 CHECK 2: Target label consistency
------------------------------------------------------------
   Total patients: 7647
   Label mismatches: 0
   ✅ PASS: Every patient has the same pneumonia_label in both files

🔍 CHECK 3: Physiological feature values match across files
------------------------------------------------------------
   ✅ heart_rate_mean: 0 mismatches out of 7647
   ✅ spo2_mean: 0 mismatches out of 7647
   ✅ respiratory_rate_mean: 0 mismatches out of 7647
   ✅ temperature_mean: 0 mismatches out of 7647
   ✅ sbp_mean: 0 mismatches out of 7647

🔍 CHECK 4: Trace 5 patients from raw labevents.csv
------------------------------------------------------------


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   Compared 991 patients
   Our wbc_value = MIN(wbc): 179
   Our wbc_value > MIN(wbc): 812
   Our wbc_value < MIN(wbc): 0
   ✅ PASS: Our wbc_value is never below the true minimum
           (first-value is always >= min — logically consistent)

✅ VERIFICATION COMPLETE
